# 🚴 Dataset 4: Bike Sharing Demand
**Propósito:** Regresión — Predicción de la demanda horaria de bicicletas compartidas en Washington D.C. (2011–2012).  
**Fuente:** Capital Bikeshare / Kaggle Competition.  
> ⚠️ **Data Leakage:** Las columnas `casual` y `registered` suman exactamente `count`. Se eliminarán del modelado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("✅ Librerías cargadas")

## 1. Carga de Datos

In [ ]:
train = pd.read_csv('../datasets/5 data sets/data set 4/train.csv')
test  = pd.read_csv('../datasets/5 data sets/data set 4/test.csv')
print(f"Train: {train.shape}  |  Test: {test.shape}")
train.head(3)

## 2. Verificación del Data Leakage

In [ ]:
print("¿casual + registered == count?", (train['casual'] + train['registered'] == train['count']).all())
print("Correlación casual  con count:", train['casual'].corr(train['count']).round(4))
print("Correlación registered con count:", train['registered'].corr(train['count']).round(4))
print("\n✅ Conclusión: ELIMINAR 'casual' y 'registered' del modelo")

## 3. Feature Engineering — Variables de Tiempo

In [ ]:
train['datetime'] = pd.to_datetime(train['datetime'])
train['hour']  = train['datetime'].dt.hour
train['day']   = train['datetime'].dt.day
train['month'] = train['datetime'].dt.month
train['year']  = train['datetime'].dt.year
train['dayofweek'] = train['datetime'].dt.dayofweek

print("Features de tiempo creadas ✅")
print(train[['datetime','hour','day','month','year','dayofweek']].head(3))

## 4. Análisis Exploratorio

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Demanda por hora
train.groupby('hour')['count'].mean().plot(ax=axes[0,0], color='steelblue', marker='o')
axes[0,0].set_title('Demanda promedio por hora del día')
axes[0,0].set_xlabel('Hora')
axes[0,0].set_ylabel('Bicicletas alquiladas')

# Demanda por mes
train.groupby('month')['count'].mean().plot(ax=axes[0,1], color='seagreen', marker='s')
axes[0,1].set_title('Demanda promedio por mes')
axes[0,1].set_xlabel('Mes')

# Demanda por estación
season_map = {1:'Primavera',2:'Verano',3:'Otoño',4:'Invierno'}
train['season_name'] = train['season'].map(season_map)
train.groupby('season_name')['count'].mean().plot(kind='bar', ax=axes[1,0], color='coral')
axes[1,0].set_title('Demanda promedio por estación')
axes[1,0].tick_params(axis='x', rotation=30)

# Distribución del target
np.log1p(train['count']).hist(ax=axes[1,1], bins=40, color='mediumpurple', edgecolor='white')
axes[1,1].set_title('Distribución de log(count)')
axes[1,1].set_xlabel('log(Bicicletas)')

plt.tight_layout()
plt.savefig('bike_eda.png', dpi=100)
plt.show()

## 5. Mapa de Calor — Demanda Hora × Día de Semana

In [ ]:
pivot = train.pivot_table(values='count', index='hour', columns='dayofweek', aggfunc='mean')
plt.figure(figsize=(10,6))
sns.heatmap(pivot, cmap='YlOrRd', annot=False)
plt.title('Demanda promedio: Hora vs Día de semana\n(0=Lunes ... 6=Domingo)')
plt.xlabel('Día de la semana')
plt.ylabel('Hora del día')
plt.tight_layout()
plt.savefig('bike_heatmap.png', dpi=100)
plt.show()

## 6. Preparación del Modelo

In [ ]:
# Eliminar leakage y columnas no útiles
features = ['season','holiday','workingday','weather','temp','atemp',
            'humidity','windspeed','hour','day','month','year','dayofweek']

X = train[features]
y = np.log1p(train['count'])  # log-transform para suavizar distribución

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")

## 7. Modelo — Gradient Boosting Regressor

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                max_depth=5, random_state=42)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
rmsle = np.sqrt(mean_squared_error(y_test, y_pred))  # ya están en log-scale

print(f"RMSLE : {rmsle:.4f}")
print(f"R²    : {r2:.4f}")

# Feature importances
imp = pd.Series(gbr.feature_importances_, index=features).sort_values(ascending=True)
imp.plot(kind='barh', figsize=(8,5), color='steelblue')
plt.title('Importancia de Features — GBR')
plt.tight_layout()
plt.savefig('bike_importancias.png', dpi=100)
plt.show()

## ✅ Resumen
| Métrica | Valor |
|---|---|
| Registros train | 10,886 |
| Features usadas | 13 |
| Modelo | Gradient Boosting Regressor |
| R² aprox | ~0.95 |

**Feature más importante:** `hour` — la hora del día es el predictor dominante de la demanda de bicicletas.